In [44]:
# from werkzeug.utils import secure_filename
import os
print(os.getcwd())
os.chdir("/Users/yuxiao/Documents/lexintel")
print(os.getcwd())
import re
from backend.util.analytics import analytics_record
from backend.ai.complaint_parser import extracted2kv
from backend.ai.base import extract_parties_venue, num_tokens_from_string, split_into_chunks
from backend.ai.base import chat_completion, chat_completion_j, extract_headings, get_coa_loc, get_fa_loc, get_venue_text, multichat_completion_j
from backend.util.db import BaseMemory, DBMemory
from backend.constants import API_URL, STATIC_FOLDER, SOURCE_CODE_ROOT, PYTHON_EXECUTABLE
from backend.util.data_tools import remove_empty_rows, selected_answers2text, allowed_filetypes, get_complaint_text_from_file, process_is_federal_output, extract_from_json, find_USC_citation, find_state_codes, remove_biz_type

# from celery import shared_task
from backend.database import db
from backend.pdf_creator import create_answer_pdf
from backend.word_creator import create_answer_word
import json
import subprocess
from backend.util.subprocess_tools import any_process_remaining
from time import sleep
from backend.util.free_law_tools import find_citations, unpack_cluster
from pprint import pprint

/Users/yuxiao/Documents/lexintel
/Users/yuxiao/Documents/lexintel


In [45]:
DOC_TYPE = "motion to dismiss"
# LINES_PER_CHUNK_PDF = 40
LINES_PER_CHUNK_DOCX = 8


def launch_llm_call_process_j(prompt:str) -> subprocess.Popen:
    """ send out multiples JSON llm calls to subprocesses """
    os.environ['PYTHONPATH'] = SOURCE_CODE_ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
    process = subprocess.Popen([PYTHON_EXECUTABLE,
                             'backend/ai/llm_call_j.py',
                             prompt],
                             stdout=subprocess.PIPE,
                             stderr=subprocess.PIPE)
    return process



def launch_llm_call_process_j_id(prompt:str, id:str) -> subprocess.Popen:
    # launch subprocess llm_call.py
    os.environ['PYTHONPATH'] = SOURCE_CODE_ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')

    process = subprocess.Popen([PYTHON_EXECUTABLE,
                             'backend/ai/llm_call_j_id.py',
                             prompt, id],
                             stdout=subprocess.PIPE,
                             stderr=subprocess.PIPE,
                            text=True)
    return process

In [46]:
# filename = "jl/JCOC v. US (LCCRSF).pdf"
# filename = "jl/JL Complaint - State Court (Ferguson).pdf"
# filename = "Gallagher_v_Edgewood_1st_Complaint.pdf"
# filename = "Meta - COMPLAINT.pdf"
# filename = "AAFER v Fearless (COMPLAINT).pdf"
# filename = "COMPLAINT-DigitalEnvoyv.Google.pdf"
filename = "COMPLAINT-Waymo.pdf"
# filename = "Tarantino_v_Gawker.pdf"
# filename = "jl/Goddard Complaint.pdf"
# filename = "musk v openai complaint.pdf"
complaint_text, case_number, is_image = get_complaint_text_from_file(STATIC_FOLDER + "/" + filename)

print("case number: ", case_number)
# memory['inputs/case_number'] = case_number

    # extract plaintiff and defendant from complaint
f_type = filename.rsplit('.', 1)[1].lower()
if f_type == 'pdf': # pdf split by special token, (marks first page), instead of hard-coded 40 lines
    cover_page_text, complaint_body_text = complaint_text.split("<CoverPage>")
    print("cover_page_text: \n", cover_page_text)
    print("complaint_body_text: \n", complaint_body_text[:100])
    # memory['complaint_text'] = complaint_body_text
    # memory['complaint_cover_text'] = first_page_text
else:
    cover_page_text = ' '.join(complaint_text.split('\n')[:LINES_PER_CHUNK_DOCX])
    complaint_body_text = complaint_text
    # memory['complaint_text'] = complaint_text   # whole doc 
    # memory['complaint_cover_text'] = first_page_text

# text_by_lines = memory['complaint_text'].split('\n')   # main body text
text_by_lines = complaint_body_text.split('\n')

print("# TOKENS (extract_parties_venue) = ", num_tokens_from_string(cover_page_text))
plaintiff_l, defendant_l, venue = extract_parties_venue(cover_page_text)
print("plaintiff: ", plaintiff_l)
print("# plaintiff = ", len(plaintiff_l))
print("defendant: ", defendant_l)
print("# defendant = ", len(defendant_l))
print("venue: ", venue)

footer:   
footer:   
footer:   
footer:  Case No._________
footer:  COMPLAINT
footer:   
header:  Case 3:17-cv-00939-WHA   Document 1   Filed 02/23/17   Page 1 of 28
*cover to add: 
 1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
 
QUINN EMANUEL URQUHART & SULLIVAN, LLP 
   Charles K. Verhoeven (Bar No. 170151) 
   charlesverhoeven@quinnemanuel.com 
   David A. Perlson (Bar No. 209502) 
   davidperlson@quinnemanuel.com 
   Melissa Baily (Bar No. 237649) 
   melissabaily@quinnemanuel.com 
   John Neukom (Bar No. 275887) 
   johnneukom@quinnemanuel.com 
   Jordan Jaffe (Bar No. 254886) 
   jordanjaffe@quinnemanuel.com 
50 California Street, 22 nd  Floor 
San Francisco, California 94111-4788 
Telephone: 
(415) 875-6600 
Facsimile: 
(415) 875-6700 
Attorneys for WAYMO LLC 
UNITED STATES DISTRICT COURT 
NORTHERN DISTRICT OF CALIFORNIA, SAN FRANCISCO DIVISION 
WAYMO LLC, 
Plaintiff, 
vs. 
UBER TECHNOLOGIES, INC.; 
OTTOMOTTO LLC; OTTO TRUCKING 
LLC, 
Defendants. 


In [47]:
# determine if this is a federal or state case
comand_prompt = open('backend/ai/prompts/answer/DETERMINE_FEDERAL').read()
filled_in_prompt = comand_prompt + venue
llm_output =  chat_completion(filled_in_prompt, model='gpt-4o-mini')
is_federal = process_is_federal_output(llm_output)
print("is_fedenral: ", is_federal)
case_info_str = f"Plaintiff: {plaintiff_l}\nDefendants: {defendant_l}\nVenue: {venue}\n Is federal: {is_federal}\n"
# memory['inputs/is_federal'] = is_federal

# extract FA from complaint as text lines
start_to_end_i = get_fa_loc(complaint_body_text)
fa_lines = text_by_lines[start_to_end_i[0] : start_to_end_i[-1]]
print(f"*** fa begins: ***\n{fa_lines[:5]}")
print(f"*** fa ends: ***\n{fa_lines[-5:]}")
# memory["fa_loc"] = ','.join([str(start_i), str(end_i)])


# split fa lines into chunk of fa lines, with 5 overlapping lines
fa_chunks = split_into_chunks(fa_lines)
print("# fa chunks: ", len(fa_chunks))

# get key information from each chunk["fa"] in fa_chunks, combine llm output to get a complete detailed summary
summarize_command = open("backend/ai/prompts/element_checker/FA_SUM").read()
fa_sums = []
processes = []
for i, chunk in enumerate(fa_chunks):
    filled_in_prompt = summarize_command.format(fa_sub=' '.join(chunk))
    # if i == 0 or i == len(fa_chunks)-1: print("*** fa sub-summary filled in prompt ***\n", filled_in_prompt)
    processes.append(launch_llm_call_process_j(filled_in_prompt))
print(f"finished launching {len(processes)} fa-sub-sum processes")
while any_process_remaining(processes):
    sleep(1)
# pack all the LLM results together
for proc in processes:
    returned = proc.stdout.read()
    each_sum_list = extract_from_json("summary", returned)
    for each in each_sum_list:
        print("llm each sum type: ", type(each))
        if type(each) == dict:
            print(each)
    fa_sums.extend(each_sum_list)
# memory['inputs/fa_sums'] = fa_sums

# send aggregated details to get final factual background
fact_bg_command = open("backend/ai/prompts/dismiss/SUGGEST_FACT_BG").read()
filled_in_prompt = fact_bg_command.format(summary='\n'.join(fa_sums))
output = chat_completion_j(filled_in_prompt)
llm_fa_bg = extract_from_json("background", output)
print(f"=== fa background ===\n", llm_fa_bg)

is_fedenral:  FEDERAL
get_fa_filled_in_prompt:  	(0)
I.	(5)
INTRODUCTION	(6)
II.	(151)
PARTIES	(152)
III.	(182)
JURISDICTION, VENUE & INTRADISTRICT ASSIGNMENT	(183)
IV.	(208)
FACTUAL ALLEGATIONS	(209)
A.	(210)
Google Pioneers The Self-Driving Car Space	(211)
B.	(263)
Waymo Develops Its Own Proprietary LiDAR System Tailored For Mass-	(264)
Marketed Self-Driving Cars	(265)
C.	(355)
Uber Is Late To Enter The Self-Driving Car Market	(356)
D.	(380)
Unbeknownst To Waymo, Anthony Levandowski Lays The Foundation For	(381)
Defendants To Steal Waymo’s Intellectual Property Rather Than Compete	(382)
Fairly In The Autonomous Vehicle Space	(383)
E.	(453)
Otto Continues To Misappropriate Waymo’s Intellectual Property After Its	(454)
Public Launch With Mr. Levandowski At The Helm	(455)
F.	(489)
After Only Six Months Of Official Existence, Otto Is Acquired By Uber For	(490)
More Than Half A Billion Dollars	(491)
G.	(505)
Waymo Verifies Its Growing Suspicion That Otto And Uber Stole Its	(506)
Intellect

In [ ]:
# # memory['inputs/coa_sums'] = coa_sum_l
# coa_titles = '\n'.join([title for title in coa_dict.keys()])
# print("== AI identified COA titles ==\n", coa_titles)
# case_info_str += f"Claims Asserted in the Complaint:\n{coa_titles}"
# save title -> {'description':desc} to database

legal_std_comand = open("backend/ai/prompts/dismiss/SUGGEST_STANDARD").read()
filled_in_prompt = legal_std_comand.format(case_info=case_info_str, fact='\n'.join(fa_sums))
#     filled_in_prompt = legal_std_comand.format(case_info=case_info_str, fact=llm_fa_bg)
#     # print("== filled_in_prompt==\n", filled_in_prompt)
#     output = chat_completion_j(filled_in_prompt)
#     llm_legal_std = extract_from_json('legal standard', output)
#     print("=== llm_legal_std ====: \n", llm_legal_std)
#     # memory['inputs/legal_standard'] = llm_legal_std

#     # output state0 here, 
#     # ask for user edit on background and legal standard

In [ ]:
# ask for user edit on background and legal standard
#     # ======================   state1 begins:  ( draft background and legal section)     =====
#     user_fa = '''....'''
#     user_standard = '''....'''

#     # prev_prompt = open("backend/ai/prompts/dismiss/SUGGEST_FACT_BG").read().format(summary='\n'.join(fa_sums))
#     # prev_output = llm_fa_bg
#     # new_prompt = open("backend/ai/prompts/dismiss/DRAFT_FACT_BG").read().format(user_input=user_fa)
#     # print("DRAFT_FACT_BG new prompt: \n", new_prompt)
#     # print("# token DRAFT_FACT_BG = ", num_tokens_from_string(prev_prompt+prev_output+new_prompt))
#     # out = multichat_completion_j(prev_prompt, prev_output, new_prompt)
#     # bg_llm_out = extract_from_json('background', out)
#     # print("== 2nd bg_llm_out ==\n", bg_llm_out)

#     # prev_prompt = open("backend/ai/prompts/dismiss/SUGGEST_STANDARD").read().format(case_info=case_info_str, fact=llm_fa_bg)
#     # prev_output = llm_legal_std
#     # new_prompt = open("backend/ai/prompts/dismiss/DRAFT_STANDARD").read().format(user_input=user_standard)
#     # print("DRAFT_STANDARD new prompt: \n", new_prompt)
#     # print("# token DRAFT_STANDARD = ", num_tokens_from_string(prev_prompt+prev_output+new_prompt))
#     # out = multichat_completion_j(prev_prompt, prev_output, new_prompt)
#     # std_llm_out = extract_from_json('legal standard', out)
#     # print("== 2nd standard_llm_out ==\n", std_llm_out)

#     # end of state1, output drafted background and legal standard

In [48]:
# ============================  state2 begins: suggest element analysis and legal ground for each claim  =================

# ========  extract COA from complaint as list of text lines  ======
# start_to_end_i = get_coa_loc(complaint_text)
start_to_end_i = get_coa_loc(complaint_body_text)
coa_lines_blocks = []
for i in range(len(start_to_end_i)-1):
    start_i = start_to_end_i[i]
    end_i = start_to_end_i[i+1]
    coa_lines_blocks.append(text_by_lines[start_i: end_i])
print("number of coa_blocks: ", len(coa_lines_blocks))
# memory["coa_loc"] = ','.join(map(str, start_to_end_i))

coa_sum_comand = open("backend/ai/prompts/dismiss/COA_SUM").read()
processes = []
for i, block in enumerate(coa_lines_blocks):
    filled_in_prompt = coa_sum_comand.format(coa=' '.join(block))
    # if i == 0 or i == len(coa_lines_blocks)-1:print("*** coa sum filled in prompt ***\n", filled_in_prompt)

    processes.append(launch_llm_call_process_j_id(filled_in_prompt, id=str(i)))
print(f"finished launching {len(processes)} coa sum processes")
while any_process_remaining(processes):
    sleep(1)
# pack all the LLM results together
print("all COA processes finished")
coa_dict = dict()
for proc in processes:
    try:
        proc_output = json.loads(proc.stdout.read())   # launch_llm_call_process_j_id return json object [id, output], but read as string
        id = int(proc_output[0])
        print("decoding id: ", id)
        claim_extracted = extract_from_json("output", proc_output[1])
        if claim_extracted:
            for each in claim_extracted:
                coa_name = each['cause of action']
                coa_desc =each['description']
                print("llm coa_name: ", coa_name)
                print("llm description: ", coa_desc)
                coa_dict[coa_name] = {'description':coa_desc}
        
    except Exception as e:
        print(f"Error decoding output: ", e)
        print(f"subprocess llm output: ", proc_output[1])
        

get_coa_filled_in_prompt:  	(0)
I.	(5)
INTRODUCTION	(6)
II.	(151)
PARTIES	(152)
III.	(182)
JURISDICTION, VENUE & INTRADISTRICT ASSIGNMENT	(183)
IV.	(208)
FACTUAL ALLEGATIONS	(209)
A.	(210)
Google Pioneers The Self-Driving Car Space	(211)
B.	(263)
Waymo Develops Its Own Proprietary LiDAR System Tailored For Mass-	(264)
Marketed Self-Driving Cars	(265)
C.	(355)
Uber Is Late To Enter The Self-Driving Car Market	(356)
D.	(380)
Unbeknownst To Waymo, Anthony Levandowski Lays The Foundation For	(381)
Defendants To Steal Waymo’s Intellectual Property Rather Than Compete	(382)
Fairly In The Autonomous Vehicle Space	(383)
E.	(453)
Otto Continues To Misappropriate Waymo’s Intellectual Property After Its	(454)
Public Launch With Mr. Levandowski At The Helm	(455)
F.	(489)
After Only Six Months Of Official Existence, Otto Is Acquired By Uber For	(490)
More Than Half A Billion Dollars	(491)
G.	(505)
Waymo Verifies Its Growing Suspicion That Otto And Uber Stole Its	(506)
Intellectual Property	(507)
H.

In [49]:
# retrieve coa_dict from database
element_command = open("backend/ai/prompts/dismiss/EVAL_ELEMENT").read()
for i, (key_coa, val) in enumerate(coa_dict.items()): 
    print("*** Start Element step for key coa", key_coa)
    filled_in_prompt = element_command.format(
        case_info=case_info_str, 
        fact=fa_sums, 
        coa = '\n'.join([key_coa, val['description']])
    )
    if i == 0:
        print("== EVAL_ELEMENT filled_in_prompt==\n", filled_in_prompt)
    output = chat_completion_j(filled_in_prompt)
    llm_elements = extract_from_json('legal elements', output)
    print("# llm_elements: ", len(llm_elements))

    coa_dict[key_coa]['elements'] = llm_elements

    # print("*******")
    # print("coa_dict with elements:")
    # print(coa_dict)
    # print("*******")


ground_command = open("backend/ai/prompts/dismiss/SUGGEST_GROUND").read()
for i, (key_coa, val) in enumerate(coa_dict.items()): 
    print("*** Start grounds for coa: ", key_coa)
    filled_in_prompt = ground_command.format(
        case_info=case_info_str, 
        fact = fa_sums,
        coa='\n'.join([key_coa, val['description']]), 
        elements=json.dumps(val['elements'])
    )
    if i == 0:
        print("== SUGGEST_GROUND filled_in_prompt ==\n", filled_in_prompt)
    returned = chat_completion_j(filled_in_prompt)
    output_grounds = extract_from_json('legal grounds', returned)
    print("1st round ground output #: ", len(output_grounds))
    coa_dict[key_coa]['legal grounds'] = output_grounds
    print(f"** Legal Ground output for {key_coa} **\n{output_grounds}")

*** Start Element step for key coa Violation of Defense of Trade Secret Act
== EVAL_ELEMENT filled_in_prompt==
 You are a senior litigator representing the defendant in the following case. You are preparing motion to dismiss to the below complaint.
You will be provided with case information and some parts of cause of action from the original complaint.
Your task is to identify the cause of action, list each legal element required to establish this cause of action under the jurisdiction identified in the complaint.
And assess this claim with matched allegations, and look for potential deficiencies.

== case information ==
Plaintiff: ['WAYMO LLC']
Defendants: ['UBER TECHNOLOGIES, INC.', 'OTTOMOTTO LLC', 'OTTO TRUCKING LLC']
Venue: UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF CALIFORNIA, SAN FRANCISCO DIVISION
 Is federal: FEDERAL


== factual allegations ==
['Google initiated its self-driving car project in 2009. (Compl. ¶22)', "Google's self-driving cars completed over 140,000 mile

In [50]:
# ====== OPTION II: SUGGEST BATCH AND THEN DRAFT ARGUMENT ====
batch_comand = open("backend/ai/prompts/dismiss/SUGGEST_BATCHES").read()
coa_grounds_str = ""
for key_coa, val in coa_dict.items():
    ground_str = ""
    for each in val['legal grounds']:
        ground_str += f"- Legal ground: {each['legal ground']}\n"
    coa_grounds_str += f"\nCause of action: {key_coa}\n" + ground_str

filled_in_prompt = batch_comand.format(
    num_coas=len(coa_dict), 
    case_info = case_info_str,
    fact = fa_sums,
    coas = coa_grounds_str
    )
print("== SUGGEST_BATCHES filled in prompt ===\n", filled_in_prompt)
returned = chat_completion_j(filled_in_prompt)
batches = extract_from_json('batches', returned)
print("# batches returned: ", len(batches))
for each in batches:
    pprint(each)

== SUGGEST_BATCHES filled in prompt ===
 You are preparing a motion to dismiss for a case involving 6 claims. Each claim may have overlapping or distinct legal grounds for dismissal. 
Below is case detials and a list of cause of actions:

== case Information
Plaintiff: ['WAYMO LLC']
Defendants: ['UBER TECHNOLOGIES, INC.', 'OTTOMOTTO LLC', 'OTTO TRUCKING LLC']
Venue: UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF CALIFORNIA, SAN FRANCISCO DIVISION
 Is federal: FEDERAL


== factual allegations
['Google initiated its self-driving car project in 2009. (Compl. ¶22)', "Google's self-driving cars completed over 140,000 miles by navigating from the Bay Area to Los Angeles, crossing the Golden Gate Bridge, driving the Pacific Coast Highway, and circling Lake Tahoe. (Compl. ¶22)", 'Google publicly announced its self-driving car project in 2010, highlighting goals of improving car safety, freeing up time, and reducing carbon emissions. (Compl. ¶23)', 'In 2014, Google unveiled a two-door fully 

In [51]:
def fas_cases_2_str(returned_j):
    returned_str = ""
    for key_coa, val in returned_j.items():
        val_str = ""
        for k, v in val.items():
            v_str = '\n'.join([each_v for each_v in v])
            s_v_str = f'{k}: \n{v_str}\n'
            val_str += s_v_str
        returned_str += f'Claim: ' + key_coa + '\n' + val_str
    return returned_str

In [52]:

 # for each batch (group of coas),  drafting a section of argument by making sequential calls, step by step
section_fa_comand = open("backend/ai/prompts/dismiss/SUGGEST_BATCH_FA").read()
section_cases_comand = open("backend/ai/prompts/dismiss/SUGGEST_BATCH_CASES").read()
# draft a section for every batch
for batch_i , each_batch in enumerate(batches):
    print("batch:")
    batch_coa_ground_element_str = ""
    # for each coa+ground (list) in a single batch, look for description and element for that coa+ground in coa_dict
    for each_coa_ground in each_batch['batch']:
        each_coa = each_coa_ground['cause of action']
        each_ground = each_coa_ground['legal ground']
        batch_coa_ground_element_str += f"\nCause of Action: {each_coa}\nLegal Ground: {each_ground}\n"
        # print("so far batch_coa_ground_element_str: ", batch_coa_ground_element_str)
        
        try:
            # print("look for this coa_ground")
            # recall coa is the key in coa_dict
            coa_all_grounds = coa_dict[each_coa]['legal grounds']  # a list of grounds for this coa
            # print("coa_all_grounds: ", coa_all_grounds)
            for coa_each_ground in coa_all_grounds:  # coa_each_ground is a dict with description, element, citation
                if each_ground == coa_each_ground['legal ground']: # look for this particular coa+ground in this batch
                    # print("found element")
                    target_ground_elem_str = f"{coa_each_ground['description']}\nDeficient Element: {coa_each_ground['element']}\nCitations: {coa_each_ground['citations']}\n"  # turn dict to str
                    batch_coa_ground_element_str += target_ground_elem_str
        except Exception as ex:
            print("Exception in Searching for coa+ground: ", str(ex))
            continue             
    print("batch_coa_ground_element_str:\n", batch_coa_ground_element_str)

    # extract relevant facts for batch
    batch_heading = each_batch['batch heading']

    section_cases_comand_filled = section_cases_comand.format(
        batch_heading = batch_heading,
        batch_coa_ground = batch_coa_ground_element_str,
        facts = '\n'.join(fa_sums)
        )
    # if batch_i == 0: 
    #     print("== SUGGEST_section_cases_comand Filled in prompt ==\n", section_cases_comand_filled)

    returned = chat_completion_j(section_cases_comand_filled)
    returned_cases = extract_from_json('cases and laws', returned)
    returned_case_str = fas_cases_2_str(returned_cases)
    # print("** BATCH FA & CASES String:\n", returned_case_str)


    section_comand_2 = open("backend/ai/prompts/dismiss/DRAFT_SECTION").read()
    section_comand_2_filled = section_comand_2.format(
        batch_heading = batch_heading,
        batch_coa_ground = batch_coa_ground_element_str,
        facts = '\n'.join(fa_sums),
        cases = returned_case_str,
    )
    returned = chat_completion_j(section_comand_2_filled, temp=0.5)
    sec_heading = extract_from_json('revised section heading', returned)
    print("SECTION HEADING: \n", sec_heading)
    arg_l = extract_from_json('section argument', returned)
    print("SECTION ARGUMENT:")
    for para in arg_l:
        print(para)

    batch_facts = []
    for i,chunk in enumerate(fa_chunks):
        section_fa_comand_filled = section_fa_comand.format(
            batch_heading = batch_heading,
            batch_coa_ground = batch_coa_ground_element_str,
            facts = '\n'.join(chunk)
        )
        llm_returned = chat_completion_j(section_fa_comand_filled)
        rel_fa_l = extract_from_json('Relevant Facts', llm_returned)
        batch_facts.extend(rel_fa_l)

    revise_command = open("backend/ai/prompts/dismiss/REVISE_SECTION").read()
    revise_command_filled = revise_command.format(
        batch_argument = f"SECTION HEADING:\n{sec_heading}\nARGUMENT:\n" + '\n'.join(arg_l),
        facts = '\n'.join(batch_facts)
    )
    returned = chat_completion_j(revise_command_filled, temp=0.5)
    sec_heading = extract_from_json('revised section heading', returned)
    print("REVISED HEADING: \n", sec_heading)
    arg_l = extract_from_json('section argument', returned)
    print("REVISED ARGUMENT:")
    for para in arg_l:
        print(para)






batch:
Exception in Searching for coa+ground:  'element'
Exception in Searching for coa+ground:  'element'
Exception in Searching for coa+ground:  'element'
batch_coa_ground_element_str:
 
Cause of Action: Violation of Defense of Trade Secret Act
Legal Ground: Failure to adequately define the trade secrets
The complaint may be dismissed on the grounds that Waymo failed to adequately define the trade secrets. The specificity of what constitutes a trade secret is crucial under the Defend Trade Secrets Act (DTSA). If the information alleged as trade secrets overlaps significantly with what is public knowledge or is too broadly described, it may not meet the DTSA's requirements for protection. A vague or overly general description undermines the element of 'Existence of a trade secret' necessary to survive a motion to dismiss.
Deficient Element: ['Existence of a trade secret']
Citations: ['18 U.S.C. § 1839(3)', 'Imax Corp. v. Cinema Techs., Inc., 152 F.3d 1161, 1164-65 (9th Cir. 1998)']

C